In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science'])
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams.update({'font.size': 10})
colors = plt.colormaps['tab20']

In [2]:
bound = lambda k, mu, s0: upper(k,mu, s0) - lower(k,mu,s0)

def upper(k,mu, s0, m=None, tol=1e-20, maxfloat=1e6):
    c = k*(1+mu)
    assert np.array(c).squeeze().ndim == 0 or np.array(s0).squeeze().ndim == 0
    if np.array(c).squeeze().ndim > 0:
        assert c.ndim == 1
        return np.array([upper(a, s0, m, tol) for a in c])
    if np.array(s0).squeeze().ndim > 0:
        assert s0.ndim == 1
        return np.array([upper(c, a, m, tol) for a in s0])
    if s0 < 0:
        return np.nan
    r = 0
    t = 1
    i = -1
    overflowcount = 0
    while True:
        i += 1
        if i > 0:
            t *= s0/i
        a = t * (c + s0)/(c + i)
        if i > 0 and np.abs(a/r) < tol:
            break
        r += a
        if m is not None and i >= m:
            break
        if r > maxfloat:
            r /= maxfloat
            t /= maxfloat
            overflowcount += 1
    r = np.exp(np.log(r) + overflowcount*np.log(maxfloat) - s0)
    return r

def lower(k,mu, s0):
    return (k*(1+mu) + s0)/(k+1+(k-1)*mu + s0)

In [3]:
def calc_k_mu_s0(p,y_hat):
    if np.sum(y_hat) == 0:
        print("Null Output")
        return None, None, None
    k = np.sum(y_hat)
    mu = np.sum(y_hat*p)/k
    s0 = np.sum((1-y_hat)*p)
    return k, mu, s0

In [4]:
from collections import defaultdict
dict_lower_bounds = defaultdict(list)
dict_upper_bounds = defaultdict(list)
dict_bounds = defaultdict(list)
dict_delta_bounds = defaultdict(list)
dict_epsilon_bounds = defaultdict(list)

In [5]:
import pickle
### MSWML
tresh = 0.35

import pickle

with open('./mswml/dev_out_predictions.pkl', 'rb') as f:
    p_hat = pickle.load(f)

with open('./mswml/dev_out_gt.pkl', 'rb') as f:
    y = pickle.load(f)
    
print(len(p_hat))

list_zeros = []
for i,p_h in enumerate(p_hat):
    binary_p_hat =  p_h>0.35
    if np.sum(binary_p_hat)==0:
        list_zeros.append(i)
        
for i in list_zeros:
    p_hat.pop(i)
    y.pop(i)
    
print(len(p_hat))



for p_hat_i in p_hat:
    p_hat_i = np.array(p_hat_i)
    y_hat_i = p_hat_i>tresh
    k, mu, s0 = calc_k_mu_s0(p_hat_i,y_hat_i)
    if (k != None) and (mu!= None) and (s0!= None):
        dict_lower_bounds['MSWML'].append(lower(k,mu, s0))
        dict_upper_bounds['MSWML'].append(upper(k,mu, s0))
        dict_delta_bounds['MSWML'].append(upper(k,mu, s0)-lower(k,mu, s0))
        dict_bounds['MSWML'].append(lower(k,mu, s0))
        dict_bounds['MSWML'].append(upper(k,mu, s0))
        dict_epsilon_bounds['MSWML'].append(max((1/lower(k,mu, s0)-1),(1-1/upper(k,mu, s0))))

25
25


In [6]:
import cv2
import os
import numpy as np
def read_images_in_directory_out(directory_path_GT):
    image_data_GT = []
    image_data_PolypPVT = []
    image_data_UNET = []
    for root, dirs, files in os.walk(directory_path_GT):
        for file in files:
            if file.endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif')) and (('CVC-ClinicDB' not in root) and ('Kvasir' not in root) and ('checkpoints' not in root)):
                image_path_GT = os.path.join(root, file)
                image_path_PolypPVT = os.path.join(root.replace('GT','PolypPVT'), file)
                image_path_UNET= os.path.join(root.replace('GT','UNET'), file)
                # Read the image in grayscale
                
                image_data_GT.append(cv2.resize(cv2.imread(image_path_GT, cv2.IMREAD_GRAYSCALE), (352, 352), interpolation = cv2.INTER_NEAREST)/255>0.5)
                image_data_PolypPVT.append(cv2.resize(cv2.imread(image_path_PolypPVT, cv2.IMREAD_GRAYSCALE), (352, 352), interpolation = cv2.INTER_NEAREST)/255)
                image_data_UNET.append(cv2.resize(cv2.imread(image_path_UNET, cv2.IMREAD_GRAYSCALE), (352, 352), interpolation = cv2.INTER_NEAREST)/255)

    
    return np.array(image_data_GT), np.array(image_data_PolypPVT), np.array(image_data_UNET)

# Specify the directory path where your images are located
directory_path_GT = './polyp/GT/Test/'
all_sm_preds = dict()
# Read the images and store them in a NumPy array and convert the list of images into a NumPy array
y, all_sm_preds['PVT'], all_sm_preds['UNET'] =read_images_in_directory_out(directory_path_GT)


In [7]:
### POLYP PVT
tresh = 0.5
y = np.expand_dims(y, axis=1)
p_hat = np.expand_dims(all_sm_preds['PVT'], axis=1)

zero_preds = np.all(y == 0, axis=(-2,-1)).flatten()
p_hat= p_hat[~zero_preds]
y = y[~zero_preds]

#binary_y_hat =  y_hat>0.5
#zero_preds = np.all(binary_y_hat == 0, axis=(-2,-1)).flatten()
#y_hat= y_hat[~zero_preds]
#y = y[~zero_preds]


print(len(y), len(p_hat))

for p_hat_i in p_hat:
    y_hat_i = p_hat_i>tresh
    k, mu, s0 = calc_k_mu_s0(p_hat_i,y_hat_i)
    if (k != None) and (mu!= None) and (s0!= None):
        dict_lower_bounds['Polyp'].append(lower(k,mu, s0))
        dict_upper_bounds['Polyp'].append(upper(k,mu, s0))
        dict_delta_bounds['Polyp'].append(upper(k,mu, s0)-lower(k,mu, s0))
        dict_bounds['Polyp'].append(lower(k,mu, s0))
        dict_bounds['Polyp'].append(upper(k,mu, s0))
        dict_epsilon_bounds['Polyp'].append(max((1/lower(k,mu, s0)-1),(1-1/upper(k,mu, s0))))

636 636


In [8]:
### POLYP UNET
'''tresh = 0.5
y = np.expand_dims(y, axis=1)
p_hat = np.expand_dims(all_sm_preds['UNET'], axis=1)

zero_preds = np.all(y == 0, axis=(-2,-1)).flatten()
p_hat= p_hat[~zero_preds]
y = y[~zero_preds]

#binary_y_hat =  y_hat>0.5
#zero_preds = np.all(binary_y_hat == 0, axis=(-2,-1)).flatten()
#y_hat= y_hat[~zero_preds]
#y = y[~zero_preds]


print(len(y), len(p_hat))

for p_hat_i in p_hat:
    y_hat_i = p_hat_i>tresh
    k, mu, s0 = calc_k_mu_s0(p_hat_i,y_hat_i)
    if (k != None) and (mu!= None) and (s0!= None):
        dict_lower_bounds['polyp-unet'].append(lower(k,mu, s0))
        dict_upper_bounds['polyp-unet'].append(upper(k,mu, s0))'''

"tresh = 0.5\ny = np.expand_dims(y, axis=1)\np_hat = np.expand_dims(all_sm_preds['UNET'], axis=1)\n\nzero_preds = np.all(y == 0, axis=(-2,-1)).flatten()\np_hat= p_hat[~zero_preds]\ny = y[~zero_preds]\n\n#binary_y_hat =  y_hat>0.5\n#zero_preds = np.all(binary_y_hat == 0, axis=(-2,-1)).flatten()\n#y_hat= y_hat[~zero_preds]\n#y = y[~zero_preds]\n\n\nprint(len(y), len(p_hat))\n\nfor p_hat_i in p_hat:\n    y_hat_i = p_hat_i>tresh\n    k, mu, s0 = calc_k_mu_s0(p_hat_i,y_hat_i)\n    if (k != None) and (mu!= None) and (s0!= None):\n        dict_lower_bounds['polyp-unet'].append(lower(k,mu, s0))\n        dict_upper_bounds['polyp-unet'].append(upper(k,mu, s0))"

In [8]:
### REFUGE
tresh = 0.5
task = 'cup'
data1 = np.load('./origa/test/%s_reshape.npz'%task)

y1 = data1['y']
y_hat1 = data1['y_hat']

zero_preds = np.all(y1 == 0, axis=(-2,-1)).flatten()
y_hat1= y_hat1[~zero_preds]
y1 = y1[~zero_preds]

#binary_y_hat1 =  y_hat1>0.5
#zero_preds1 = np.all(binary_y_hat1 == 0, axis=(-2,-1)).flatten()
#y_hat1= y_hat1[~zero_preds1]
#y1 = y1[~zero_preds1]


data2 = np.load('./G1020/test/%s_reshape.npz'%task)

y2 = data2['y']
y_hat2 = data2['y_hat']

#binary_y_hat =  y_hat>0.5
#zero_preds = np.all(binary_y_hat == 0, axis=(-2,-1)).flatten()
#y_hat= y_hat[~zero_preds]
#y = y[~zero_preds]

zero_preds = np.all(y2 == 0, axis=(-2,-1)).flatten()
y_hat2= y_hat2[~zero_preds]
y2 = y2[~zero_preds]

print(y1.shape, y_hat1.shape)
print(y2.shape, y_hat2.shape)
y = np.concatenate((y1,y2))
p_hat = np.concatenate((y_hat1,y_hat2))

y.shape, p_hat.shape

#binary_y_hat =  y_hat>0.5
#zero_preds = np.all(binary_y_hat == 0, axis=(-2,-1)).flatten()
#y_hat= y_hat[~zero_preds]
#y = y[~zero_preds]


print(len(y), len(p_hat))

for p_hat_i in p_hat:
    y_hat_i = p_hat_i>tresh
    k, mu, s0 = calc_k_mu_s0(p_hat_i,y_hat_i)
    if (k != None) and (mu!= None) and (s0!= None):
        dict_lower_bounds['REFUGE'].append(lower(k,mu, s0))
        dict_upper_bounds['REFUGE'].append(upper(k,mu, s0))
        dict_delta_bounds['REFUGE'].append(upper(k,mu, s0)-lower(k,mu, s0))
        dict_bounds['REFUGE'].append(lower(k,mu, s0))
        dict_bounds['REFUGE'].append(upper(k,mu, s0))
        dict_epsilon_bounds['REFUGE'].append(max((1/lower(k,mu, s0)-1),(1-1/upper(k,mu, s0))))

(648, 1, 575, 575) (648, 1, 575, 575)
(782, 1, 575, 575) (782, 1, 575, 575)
1430 1430
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output


In [9]:
import pandas as pd

def generate_latex_table(data):
    # Convert dictionary to DataFrame, handling unequal lengths by filling with NaN
    max_length = max(len(v) for v in data.values())
    padded_data = {key: values + [np.nan] * (max_length - len(values)) for key, values in data.items()}
    df = pd.DataFrame(padded_data)
    
    summary_stats = df.agg(['min', 'max', 'median']).transpose()
    
    # Format the values to 5 decimal places
    summary_stats = summary_stats.round(5)
    
    # Generate the LaTeX table
    latex_table = "\\begin{table}[h!]\n\\centering\n\\begin{tabular}{|l|r|r|r|}\n\\hline\n"
    latex_table += "Dataset & Min & Max & Median \\\\\n\\hline\n"
    for col in summary_stats.index:
        min_val = f"{summary_stats.loc[col, 'min']:.5f}"
        max_val = f"{summary_stats.loc[col, 'max']:.5f}"
        median_val = f"{summary_stats.loc[col, 'median']:.5f}"
        latex_table += f"{col} & {min_val} & {max_val} & {median_val} \\\\\n"
    latex_table += "\\hline\n\\end{tabular}\n\\caption{Summary Statistics Table}\n\\label{tab:summary_stats}\n\\end{table}"
    
    return latex_table

In [10]:
import numpy as np
import pandas as pd

def scientific_notation_around_one(value):
    """Format a number close to 1 in scientific notation as 1 +/- small deviation."""
    if pd.isna(value):
        return "NaN"
    deviation = value - 1
    if np.isclose(deviation, 0, atol=1e-50):
        return "$1.0$	"
    elif deviation > 0:
        base, exponent = f"{deviation:.1e}".split("e")
        return f"$1.0 + {float(base):.1f} \\times 10^{{{int(exponent)}}}$"
    else:
        base, exponent = f"{abs(deviation):.1e}".split("e")
        return f"$1.0 - {float(base):.1f} \\times 10^{{{int(exponent)}}}$"

def scientific_notation_around_zero(value):
    """Format a number close to 0 in scientific notation"""
    if pd.isna(value):
        return "NaN"
    if np.isclose(value, 0, atol=1e-50):
        return "$0.0$	"
    else:
        base, exponent = f"{value:.1e}".split("e")
        return f"${float(base):.1f} \\times 10^{{{int(exponent)}}}$"

def generate_latex_table(data, data_delta):
    # Convert dictionary to DataFrame, handling unequal lengths by filling with NaN
    max_length = max(len(v) for v in data.values())
    padded_data = {key: values + [np.nan] * (max_length - len(values)) for key, values in data.items()}
    df = pd.DataFrame(padded_data)

    max_length_delta = max(len(v) for v in data_delta.values())
    padded_data_delta = {key: values + [np.nan] * (max_length_delta - len(values)) for key, values in data_delta.items()}
    df_delta = pd.DataFrame(padded_data_delta)
    
    summary_stats = df.agg(['min', 'max', 'median']).transpose()
    summary_stats_delta= df_delta.agg(['max', 'mean']).transpose()
    
    # Generate the LaTeX table
    latex_table = "\\begin{table}[h!]\n\\centering\n\\begin{tabular}{|l|c|c|c|c|}\n\\hline\n"
    latex_table += "\\textbf{Dataset} & \textbf{Min($b_L$)} & \textbf{Max($b_U$)} & \textbf{Max($b_U - b_L$)} & \textbf{Mean($b_U - b_L$)} \\\\ \n\\hline\n"
    for col in summary_stats.index:
        min_val = scientific_notation_around_one(summary_stats.loc[col, 'min'])
        max_val = scientific_notation_around_one(summary_stats.loc[col, 'max'])
        max_val_delta = scientific_notation_around_zero(summary_stats_delta.loc[col, 'max'])
        mean_val_delta = scientific_notation_around_zero(summary_stats_delta.loc[col, 'mean'])
        latex_table += f"\\texttt{{{col}}} & {min_val} & {max_val} & {max_val_delta} & {mean_val_delta} \\\\ \n"
    latex_table += "\\hline\n\\end{tabular}\n\\caption{Summary Statistics for Each Dataset}\n\\label{tab:summary_stats}\n\\end{table}"

    return latex_table



In [11]:
print(generate_latex_table(dict_bounds,dict_delta_bounds))

\begin{table}[h!]
\centering
\begin{tabular}{|l|c|c|c|c|}
\hline
\textbf{Dataset} & 	extbf{Min($b_L$)} & 	extbf{Max($b_U$)} & 	extbf{Max($b_U - b_L$)} & 	extbf{Mean($b_U - b_L$)} \\ 
\hline
\texttt{MSWML} & $1.0 - 7.6 \times 10^{-4}$ & $1.0 + 1.4 \times 10^{-3}$ & $2.2 \times 10^{-3}$ & $1.9 \times 10^{-4}$ \\ 
\texttt{Polyp} & $1.0 - 3.4 \times 10^{-3}$ & $1.0 + 4.5 \times 10^{-3}$ & $7.8 \times 10^{-3}$ & $2.7 \times 10^{-5}$ \\ 
\texttt{REFUGE} & $1.0 - 2.4 \times 10^{-4}$ & $1.0 + 4.8 \times 10^{-4}$ & $7.2 \times 10^{-4}$ & $7.4 \times 10^{-6}$ \\ 
\hline
\end{tabular}
\caption{Summary Statistics for Each Dataset}
\label{tab:summary_stats}
\end{table}


In [12]:
import numpy as np
import pandas as pd

def scientific_notation_around_one(value):
    """Format a number close to 1 in scientific notation as 1 +/- small deviation."""
    if pd.isna(value):
        return "NaN"
    deviation = value - 1
    if np.isclose(deviation, 0, atol=1e-50):
        return "$1.0$	"
    elif deviation > 0:
        base, exponent = f"{deviation:.1e}".split("e")
        return f"$1.0 + {float(base):.1f} \\times 10^{{{int(exponent)}}}$"
    else:
        base, exponent = f"{abs(deviation):.1e}".split("e")
        return f"$1.0 - {float(base):.1f} \\times 10^{{{int(exponent)}}}$"

def scientific_notation_around_zero(value):
    """Format a number close to 0 in scientific notation"""
    if pd.isna(value):
        return "NaN"
    if np.isclose(value, 0, atol=1e-50):
        return "$0.0$	"
    else:
        base, exponent = f"{value:.1e}".split("e")
        return f"${float(base):.1f} \\times 10^{{{int(exponent)}}}$"

def generate_latex_table(data):
    # Convert dictionary to DataFrame, handling unequal lengths by filling with NaN
    max_length = max(len(v) for v in data.values())
    padded_data = {key: values + [np.nan] * (max_length - len(values)) for key, values in data.items()}
    df = pd.DataFrame(padded_data)
    
    summary_stats= df.agg(['max', 'mean']).transpose()
    
    # Generate the LaTeX table
    latex_table = "\\begin{table}[h!]\n\\centering\n\\begin{tabular}{|l|c|c|}\n\\hline\n"
    latex_table += "\\textbf{Dataset} & \\textbf{Max($\epsilon$)}  & \\textbf{Mean($\epsilon$)} \\\\ \n\\hline\n"
    for col in summary_stats.index:
        max_val = scientific_notation_around_zero(summary_stats.loc[col, 'max'])
        mean_val= scientific_notation_around_zero(summary_stats.loc[col, 'mean'])
        latex_table += f"\\texttt{{{col}}} & {max_val} & {mean_val} \\\\ \n"
    latex_table += "\\hline\n\\end{tabular}\n\\caption{Summary Statistics for Each Dataset}\n\\label{tab:summary_stats}\n\\end{table}"

    return latex_table



In [13]:
print(generate_latex_table(dict_epsilon_bounds))

\begin{table}[h!]
\centering
\begin{tabular}{|l|c|c|}
\hline
\textbf{Dataset} & \textbf{Max($\epsilon$)}  & \textbf{Mean($\epsilon$)} \\ 
\hline
\texttt{MSWML} & $1.4 \times 10^{-3}$ & $1.2 \times 10^{-4}$ \\ 
\texttt{Polyp} & $4.4 \times 10^{-3}$ & $1.5 \times 10^{-5}$ \\ 
\texttt{REFUGE} & $4.8 \times 10^{-4}$ & $4.4 \times 10^{-6}$ \\ 
\hline
\end{tabular}
\caption{Summary Statistics for Each Dataset}
\label{tab:summary_stats}
\end{table}


In [14]:
dict_epsilon_bounds

defaultdict(list,
            {'MSWML': [5.555598921569427e-06,
              0.00012421025657349816,
              0.0001976765065339059,
              3.9564123943680585e-06,
              1.452923760991709e-05,
              7.2300171982675465e-06,
              9.407785785953138e-07,
              1.789129602247641e-06,
              4.318756549714919e-06,
              5.3249715932235375e-06,
              3.1048601116800967e-06,
              2.8991689885904393e-05,
              9.662882883110235e-05,
              1.3777358882638424e-06,
              1.8895525339290842e-05,
              2.916362646487869e-06,
              3.8200725887005405e-06,
              8.733843733566005e-06,
              0.00011711641298495223,
              5.389199970062819e-06,
              8.590527499796607e-06,
              9.011006105730246e-05,
              9.521847013083118e-06,
              0.0008995361293404613,
              0.0014291098161728355],
             'Polyp': [5.551675987813

In [15]:
import pandas as pd
x = np.inf
for i in dict_upper_bounds:
    y = min(dict_upper_bounds[i])
    if y<x:
        x == y
print(y)

1.000000131673923


In [16]:
import pandas as pd
x = -np.inf
for i in dict_upper_bounds:
    y = max(dict_upper_bounds[i])
    if y>x:
        x == y
print(y)

1.000478409407655


In [17]:
import pandas as pd
x = np.inf
for i in dict_upper_bounds:
    y = min(dict_lower_bounds[i])
    if y<x:
        x == y
print(y)

0.9997587567411926


In [18]:
import pandas as pd
x = -np.inf
for i in dict_upper_bounds:
    y = max(dict_lower_bounds[i])
    if y>x:
        x == y
print(y)

0.9999996712472032
